## Lab 10: SQL Python

In this lab we will look at how to use the python library pymysql to interact with MySQL Workbench.

* We will look at how to load the contents of a .csv as a pandas dataframe, preprocess it, and populate a SQL database table with this data.
* Finally, we will see how to run queries against an existing database table using pymysql.

### Part 1: European Countries Dataset

The dataset relates to countries in Europe. It contains information such as country name, its surface area, population, GDP, its capital city, and the population of its capital city.

The goal of this project is to perform Extract Load & Transform (ELT) the dataset. We will use preprocessing to ensure the data is free of any formatting errors, and store it into a SQL database. 

In [ ]:
# Install any missing packages
#!pip install pymysql

In [4]:
#Load the required libraries
import pandas as pd
import numpy as np
import os
import pymysql

### 1. Extract (Gather the Data)

In [5]:
raw_csv_data = pd.read_csv('euro_countries.csv')

In [6]:
#make a dataframe to copy the data
df = raw_csv_data.copy()

In [7]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [8]:
display(df)

,Country,Area (miles2),Capital,Population,GDP (2016),Capital Population,Longitude,Longitude Direction,Latitude
0,'Albania',11099.66,'Tirana',2854710,1.186387e+10,49300,19.816667,'E',41.316667
1,'Andorra',180.70,'Andorra_la_Vella',79034,2.858518e+09,22256,1.516667,'E',42.500000
2,'Armenia',11483.83,'Yerevan',2790974,1.057230e+10,1100000,44.500000,'E',40.166667
3,'Austria',32385.46,'Vienna',8922082,3.910000e+11,1900000,16.366667,'E',48.200000
4,'Azerbaijan',2687.27,'Baku',10312992,3.784772e+10,2240000,49.866667,'E',40.383333
5,'Belarus',80154.78,'Minsk',9578167,4.740722e+10,2000000,27.566667,'E',53.900000
6,'Belgium',11786.92,'Brussels',11611419,4.680000e+11,180000,4.333333,'E',50.833333
7,'Bosnia and Herzegovina',19771.90,'Sarajevo',3270943,1.691028e+10,280000,18.416667,'E',43.866667
8,'Bulgaria',42614.85,'Sofia',6885868,5.323788e+10,1280000,23.316667,'E',42.683333
9,'Croatia',21851.06,'Zagreb',4060135,5.071496e+10,900000,16.000000,'E',45.800000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              49 non-null     object 
 1   Area (miles2)        49 non-null     float64
 2   Capital              49 non-null     object 
 3   Population           49 non-null     int64  
 4   GDP (2016)           49 non-null     float64
 5   Capital Population   49 non-null     int64  
 6   Longitude            49 non-null     float64
 7   Longitude Direction  49 non-null     object 
 8   Latitude             49 non-null     float64
dtypes: float64(4), int64(2), object(3)
memory usage: 3.6+ KB


### 2. Transform (Preprocess the Data)

#### 2.1 Convert the Area from squared miles in squared kilometres

We need to convert the Area from squared miles to squared kilometres.

In [10]:
df['Area (miles2)'] = df['Area (miles2)'] * 2.58999

Now, let's rename the column from Area (miles2) to Area (km2).

In [11]:
df = df.rename(columns={"Area (miles2)": "Area (km2)"})

#### 2.2 Reformat the Longitude to have "-" for locations West of Greenwich

We have a small issue with the values in the "Longitude" column. They should have a "-" for locations west of Greenwich (meridian line) and a blank (positive) for locations to the east of it. There is a second column called "Longitude Direction" which contains "W" or "E" depending on the direction w.r.t to the Greenwich line. We can use this information to modify the data in the "Longitude" column.

In [12]:
print(df['Longitude Direction'][0])

'E'


In [13]:
# If the longitude direction is West of Greenwich then multiply the values in the Longitude Column by -1
for i in range(len(df)):
    if df['Longitude Direction'][i] == "'W'":
        df['Longitude'][i] = df['Longitude'][i] * -1.00

C:\Users\RoksolanaKryvulia-ST\AppData\Local\Temp\ipykernel_15572\2905045340.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['Longitude'][i] = df['Longitude'][i] * -1.00
C:\Users\RoksolanaKryvulia-ST\AppData\Local\Temp\ipykernel_15572\

In [14]:
# Finally delete (drop) the column called 'Longitude Direction'
df = df.drop(columns=['Longitude Direction'])

#### 2.3 Review the Transformed Data

In [15]:
display(df)

,Country,Area (km2),Capital,Population,GDP (2016),Capital Population,Longitude,Latitude
0,'Albania',2.874801e+04,'Tirana',2854710,1.186387e+10,49300,19.816667,41.316667
1,'Andorra',4.680112e+02,'Andorra_la_Vella',79034,2.858518e+09,22256,1.516667,42.500000
2,'Armenia',2.974300e+04,'Yerevan',2790974,1.057230e+10,1100000,44.500000,40.166667
3,'Austria',8.387802e+04,'Vienna',8922082,3.910000e+11,1900000,16.366667,48.200000
4,'Azerbaijan',6.960002e+03,'Baku',10312992,3.784772e+10,2240000,49.866667,40.383333
5,'Belarus',2.076001e+05,'Minsk',9578167,4.740722e+10,2000000,27.566667,53.900000
6,'Belgium',3.052800e+04,'Brussels',11611419,4.680000e+11,180000,4.333333,50.833333
7,'Bosnia and Herzegovina',5.120902e+04,'Sarajevo',3270943,1.691028e+10,280000,18.416667,43.866667
8,'Bulgaria',1.103720e+05,'Sofia',6885868,5.323788e+10,1280000,23.316667,42.683333
9,'Croatia',5.659403e+04,'Zagreb',4060135,5.071496e+10,900000,16.000000,45.800000


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             49 non-null     object 
 1   Area (km2)          49 non-null     float64
 2   Capital             49 non-null     object 
 3   Population          49 non-null     int64  
 4   GDP (2016)          49 non-null     float64
 5   Capital Population  49 non-null     int64  
 6   Longitude           49 non-null     float64
 7   Latitude            49 non-null     float64
dtypes: float64(4), int64(2), object(2)
memory usage: 3.2+ KB


### 3. Load (Put the Data into a SQL Database)

### 3.1 Create a connection to MyQL

In [86]:
#Create a connection to MySQL through pymysql package

conn = pymysql.connect(database = 'EU_data', user = 'root', password = 'password')

In [87]:
cursor = conn.cursor()

### 3.2 Populate the data into the table in the database

In [88]:
insert_query = 'INSERT INTO eu_contries VALUES '

In [89]:
# Use a loop to cycle through dataframe columns and values and populate our MySQL table accordingly

# TODO - write the block of code to perform this
for i in range(df.shape[0]):
    insert_query +='('
    
    for j in range(df.shape[1]):
        insert_query +=str(df[df.columns.values[j]][i])+ ', '

    insert_query = insert_query[:-2]+'), '

In [90]:
# check the query now
insert_query

"INSERT INTO eu_contries VALUES ('Albania', 28748.008403399996, 'Tirana', 2854710, 11863865978.0, 49300, 19.816667, 41.31666667), ('Andorra', 468.01119299999993, 'Andorra_la_Vella', 79034, 2858517699.0, 22256, 1.516667, 42.5), ('Armenia', 29743.004861699996, 'Yerevan', 2790974, 10572298342.0, 1100000, 44.5, 40.16666667), ('Austria', 83878.0175454, 'Vienna', 8922082, 391000000000.0, 1900000, 16.366667, 48.2), ('Azerbaijan', 6960.002427299999, 'Baku', 10312992, 37847715736.0, 2240000, 49.866667, 40.38333333), ('Belarus', 207600.07865219997, 'Minsk', 9578167, 47407217531.0, 2000000, 27.566667, 53.9), ('Belgium', 30528.004930799998, 'Brussels', 11611419, 468000000000.0, 180000, 4.333333, 50.83333333), ('Bosnia and Herzegovina', 51209.023281, 'Sarajevo', 3270943, 16910277134.0, 280000, 18.416667, 43.86666667), ('Bulgaria', 110372.03535149999, 'Sofia', 6885868, 53237882473.0, 1280000, 23.316667, 42.68333333), ('Croatia', 56594.0268894, 'Zagreb', 4060135, 50714957391.0, 900000, 16.0, 45.8), (

In [91]:
insert_query = insert_query[:-2] + ';'

In [92]:
# check the query once more
insert_query

"INSERT INTO eu_contries VALUES ('Albania', 28748.008403399996, 'Tirana', 2854710, 11863865978.0, 49300, 19.816667, 41.31666667), ('Andorra', 468.01119299999993, 'Andorra_la_Vella', 79034, 2858517699.0, 22256, 1.516667, 42.5), ('Armenia', 29743.004861699996, 'Yerevan', 2790974, 10572298342.0, 1100000, 44.5, 40.16666667), ('Austria', 83878.0175454, 'Vienna', 8922082, 391000000000.0, 1900000, 16.366667, 48.2), ('Azerbaijan', 6960.002427299999, 'Baku', 10312992, 37847715736.0, 2240000, 49.866667, 40.38333333), ('Belarus', 207600.07865219997, 'Minsk', 9578167, 47407217531.0, 2000000, 27.566667, 53.9), ('Belgium', 30528.004930799998, 'Brussels', 11611419, 468000000000.0, 180000, 4.333333, 50.83333333), ('Bosnia and Herzegovina', 51209.023281, 'Sarajevo', 3270943, 16910277134.0, 280000, 18.416667, 43.86666667), ('Bulgaria', 110372.03535149999, 'Sofia', 6885868, 53237882473.0, 1280000, 23.316667, 42.68333333), ('Croatia', 56594.0268894, 'Zagreb', 4060135, 50714957391.0, 900000, 16.0, 45.8), (

### 3.3 Execute the query

In [93]:
# Now execute the query
cursor.execute(insert_query)

49

In [94]:
# Commit the changes to SQL connection
conn.commit()

In [95]:
conn.close()

### Part 2: Run queries against an existing database table

We will use the employees schema for this practical. Download the file employees.sql from blackboard. It consists of tables: employees, titles, salaries, departments, dept_manager. 

In [ ]:
#Create a connection to MySQL through pymysql package

conn = pymysql.connect(database = 'employees', user = 'root', password = 'password')

In [ ]:
cursor = conn.cursor()

#### TOPIC: Select, JOIN, COUNT, AVG

**Q1:** Return employee number, first name, last name and hire date of all employees with the first name Vincent (hint: WHERE). 

In [ ]:
# Question 1

cursor.execute('SELECT fisrt_name, last_name, hire_data FROM employees WHERE first_name ="Vincent";')


In [ ]:
results = cursor.fetchall()
for row in results:
    print(row)

**Q2:** Return all columns for all employees whose first name starts with “Mar..” i.e. Mark, Margaret, Markus etc. (hint: LIKE and “Letters%”). 

In [ ]:
# Question 2

#TODO:

**Q3:** Number of employees with the last name Boyle (hint: COUNT, WHERE). 

In [ ]:
# Question 3

#TODO:

**Q4:** Get the top 10 last names within the company (hint: GROUP BY, ORDER BY, LIMIT).

In [ ]:
# Question 4

#TODO:

**Q5:** How many employees were hired by the company in the whole of the year 1985? (Hint: COUNT, BETWEEN).

In [ ]:
# Question 5

#TODO:

**Q6:** Find the average salary per job title. (Hint: JOIN, GROUP BY).

In [ ]:
# Question 6

#TODO:

#### TOPIC: Nested / Subqueries

**Q7:** Using a subquery, find the first name, last name and salary of the employee with the highest salary (still at company). 

In [ ]:
# Question 7

#TODO:

**Q8:** Find the first name, last name, and job-titles of all employees with the same job-title as Georgi Facello using a sub-query in the WHERE clause of the main outer query. Hint: First find the job-title of Georgi Facello using a query, and then put that query into the WHERE clause of the main query (Hence it will act as the inner query).

Note: you’ll need to make use of JOIN and a WHERE condition within that inner query including to_date = ‘9999-01-01’. You’ll also need to use a JOIN in the outer query too.

In [ ]:
# Question 8

#TODO:

Finally close the connection

In [ ]:
conn.close()